<a href="https://colab.research.google.com/github/Park-gpow/bigdataAn/blob/main/data(03-31_02).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# 기존 설정(매 리셋마다 실행)
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import json

drive.mount('/content/drive')

BASE_DIR = Path("/content/drive/MyDrive/빅데이터 사례연구")

TRAIN_PATH = Path("/content/drive/MyDrive/빅데이터 사례연구/69.고품질 연구 개발용 이차전지 데이터/Training/TS_LFP(리튬-철-인)/LFP_train_dataset.csv")
VAL_PATH = Path("/content/drive/MyDrive/빅데이터 사례연구/69.고품질 연구 개발용 이차전지 데이터/Validation/TS_LFP(리튬-철-인)/LFP_val_dataset.csv")

TR_OUT_DIR = BASE_DIR / "Tr"
VAL_OUT_DIR = BASE_DIR / "Val"

TARGET_COL = "discharge_capacity"

HARD_EXCLUDE_COLS = [
    "material_id",
    "DOI",
    "journal_name",
    "chemical_formula",
    "remain_capacity",
    "voltage_range",
]

TARGET_CANDIDATES = [
    "discharge_capacity (mAh/g)",
    "discharge_capacity",
]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TRAIN_PATH = /content/drive/MyDrive/빅데이터 사례연구/69.고품질 연구 개발용 이차전지 데이터/Training/TS_LFP(리튬-철-인)/LFP_train_dataset.csv
VAL_PATH   = /content/drive/MyDrive/빅데이터 사례연구/69.고품질 연구 개발용 이차전지 데이터/Validation/TS_LFP(리튬-철-인)/LFP_val_dataset.csv
TRAIN 존재: True
VAL 존재  : True
       material_id material_structure synthesis_method  sintering_T1(C)  \
0  sb-qm02g6seq327            Olivine            Other           80.000   
1  sb-s0p64tj6loib            Olivine            Other          481.439   
2  sb-yqj0n0jkkxxu            Olivine            Other          800.000   
3  sb-5btluxs1d42z            Olivine          Sol-gel          850.000   
4  sb-3fuphm5i9fut            Olivine            Other          800.000   

   sintering_t1(h) Li_source     Co_source     Mn_source     Ni_source  \
0           12.000     Other  not included  not included  not included   
1           

In [ ]:
#원본 로딩 확인(시행 여부 체크)
train_raw = pd.read_csv(TRAIN_PATH)
val_raw = pd.read_csv(VAL_PATH)

print("train_raw shape:", train_raw.shape)
print("val_raw shape:", val_raw.shape)

display(train_raw.head())
display(val_raw.head())

In [ ]:
#전처리 + 저장 (한번만 실행)
TR_OUT_DIR.mkdir(parents=True, exist_ok=True)
VAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

train = train_raw.copy()
val = val_raw.copy()

rename_map_train = {}
for col in train.columns:
    clean = col.strip()
    if clean in TARGET_CANDIDATES:
        rename_map_train[col] = "discharge_capacity"
    elif clean == "voltage_range(V)":
        rename_map_train[col] = "voltage_range"
    else:
        rename_map_train[col] = clean

rename_map_val = {}
for col in val.columns:
    clean = col.strip()
    if clean in TARGET_CANDIDATES:
        rename_map_val[col] = "discharge_capacity"
    elif clean == "voltage_range(V)":
        rename_map_val[col] = "voltage_range"
    else:
        rename_map_val[col] = clean

train = train.rename(columns=rename_map_train).copy()
val = val.rename(columns=rename_map_val).copy()

if "voltage_range" in train.columns:
    vr_train = train["voltage_range"].astype(str).str.replace("~", ",", regex=False)
    parts_train = vr_train.str.extract(r"\s*([+-]?\d*\.?\d+)\s*[,/ ]+\s*([+-]?\d*\.?\d+)")
    if parts_train.notna().any().any():
        train["voltage_range_min"] = pd.to_numeric(parts_train[0], errors="coerce")
        train["voltage_range_max"] = pd.to_numeric(parts_train[1], errors="coerce")
        train["voltage_window"] = train["voltage_range_max"] - train["voltage_range_min"]

if "voltage_range" in val.columns:
    vr_val = val["voltage_range"].astype(str).str.replace("~", ",", regex=False)
    parts_val = vr_val.str.extract(r"\s*([+-]?\d*\.?\d+)\s*[,/ ]+\s*([+-]?\d*\.?\d+)")
    if parts_val.notna().any().any():
        val["voltage_range_min"] = pd.to_numeric(parts_val[0], errors="coerce")
        val["voltage_range_max"] = pd.to_numeric(parts_val[1], errors="coerce")
        val["voltage_window"] = val["voltage_range_max"] - val["voltage_range_min"]

existing_hard_exclude = [col for col in HARD_EXCLUDE_COLS if col in train.columns]

constant_cols = []
for col in train.columns:
    if col == TARGET_COL:
        continue
    if train[col].nunique(dropna=False) <= 1:
        constant_cols.append(col)

candidate_feature_cols = [
    col for col in train.columns
    if col not in existing_hard_exclude and col not in constant_cols and col != TARGET_COL
]

feature_cols = [col for col in candidate_feature_cols if col in val.columns]

numeric_cols = [col for col in feature_cols if pd.api.types.is_numeric_dtype(train[col])]
categorical_cols = [col for col in feature_cols if col not in numeric_cols]

X_train = train[feature_cols].copy()
X_val = val[feature_cols].copy()

y_train = pd.to_numeric(train[TARGET_COL], errors="coerce")
y_val = pd.to_numeric(val[TARGET_COL], errors="coerce")

train_keep = y_train.notna()
val_keep = y_val.notna()

X_train = X_train.loc[train_keep].copy()
X_val = X_val.loc[val_keep].copy()
y_train = y_train.loc[train_keep].copy()
y_val = y_val.loc[val_keep].copy()

numeric_fill_values = {}
for col in numeric_cols:
    fill_value = pd.to_numeric(X_train[col], errors="coerce").median()
    numeric_fill_values[col] = fill_value
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce").fillna(fill_value)
    X_val[col] = pd.to_numeric(X_val[col], errors="coerce").fillna(fill_value)

for col in categorical_cols:
    X_train[col] = X_train[col].astype("string").fillna("Missing")
    X_val[col] = X_val[col].astype("string").fillna("Missing")

X_train_model = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_val_model = pd.get_dummies(X_val, columns=categorical_cols, drop_first=True)

X_train_model, X_val_model = X_train_model.align(X_val_model, join="left", axis=1, fill_value=0)

X_train_model.to_csv(TR_OUT_DIR / "X_train_model.csv", index=False, encoding="utf-8-sig")
X_val_model.to_csv(VAL_OUT_DIR / "X_val_model.csv", index=False, encoding="utf-8-sig")
y_train.to_csv(TR_OUT_DIR / "y_train.csv", index=False, encoding="utf-8-sig")
y_val.to_csv(VAL_OUT_DIR / "y_val.csv", index=False, encoding="utf-8-sig")

metadata = {
    "TRAIN_PATH": str(TRAIN_PATH),
    "VAL_PATH": str(VAL_PATH),
    "target": TARGET_COL,
    "hard_excluded": existing_hard_exclude,
    "constant_excluded": constant_cols,
    "feature_cols": feature_cols,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "X_train_model_shape": list(X_train_model.shape),
    "X_val_model_shape": list(X_val_model.shape),
}
with open(TR_OUT_DIR / "preprocessing_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("전처리 저장 완료")

In [ ]:
#분석 셀
X_train_model = pd.read_csv(TR_OUT_DIR / "X_train_model.csv")
X_val_model = pd.read_csv(VAL_OUT_DIR / "X_val_model.csv")
y_train = pd.read_csv(TR_OUT_DIR / "y_train.csv").squeeze("columns")
y_val = pd.read_csv(VAL_OUT_DIR / "y_val.csv").squeeze("columns")

print(X_train_model.shape, X_val_model.shape, y_train.shape, y_val.shape)

# 여기서부터 모델링 반복